In [1]:
from modules.dataset.make_dataset import make_dataset, dataset_split_by_recording, stratified_recording_folds, NpzDataPipeline, AugmentedSubset
from modules.nn import CoralOrdinal, DefaultCNN, coral_pos_weights, coral_targets, ordinal_oof_metrics, ordinal_predict
import polars as pl
import numpy as np
import torch
from torchinfo import summary
from torch.utils.data import Subset
import torch.nn as nn
import torch.optim as optim
from typing import Sequence
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [2]:
dataset: NpzDataPipeline = NpzDataPipeline("data\\final_processed_dataset.npz")
device: torch.types.Device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
import os

# ---- Crash-proof run state: per-epoch autosave + resume (train_ordinal) ----
RUN_STATE_DIR: str = "run_state_3class"
os.makedirs(RUN_STATE_DIR, exist_ok=True)

def _ckpt_path(name: str) -> str:
    return os.path.join(RUN_STATE_DIR, f"{name}_latest.pth.tar")

def _atomic_torch_save(obj: dict, path: str) -> None:
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)  # crash mid-write never corrupts the good file


In [4]:
# ---- Experiment setup (shared) ----
from modules.dataset.make_dataset import AugmentedSubset, aug_probs_from_counts
from modules.preprocessing.audio_process import WaveformAugmentor, SpecAugmentations, MelSpectrogramPipeline, SpectMFCCPipeline
from tqdm.auto import tqdm

num_epochs: int = 60  # epoch cap for ALL runs; early stop (15/3) lives in train_ordinal
N_FOLDS: int = 5
SEED: int = 42  # identical split across all experiments
BATCH_SIZE: int = 64

# n_mel=64 per Chang et al. (research_results.md fix #6): 64 mel bins @ 16 kHz
mel_train_plain: SpectMFCCPipeline = SpectMFCCPipeline(combind=True).to(device)  # no spec aug
mel_eval: SpectMFCCPipeline = SpectMFCCPipeline(combind=True).to(device)         # plain for validation

# speed: compile only the eval mel path; train paths stay eager (spec-aug masks/augmentor randomize per step)
mel_eval = torch.compile(mel_eval) if device.type == "cuda" else mel_eval  # CPU inductor needs MSVC cl.exe; skip
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


In [5]:
# ---- Recording-wise 5-fold CV setup (replaces the single 80/10/10 split) ----
# Recording-grouped, rank-stratified folds. The old test split survives only as
# the pool definition: train+val recordings form the CV pool, test stays held out.
# Every reported metric comes from pooled out-of-fold predictions, aggregated to
# the RECORDING level before anything is computed.

dataset.load_data()
dataset.load_paths()

train_subset, val_subset, test_subset = dataset_split_by_recording(dataset, dataset.paths, 0.8, 0.1, seed=SEED)

# rank mapping: healthy=low(0), symptomatic=med(1), COVID-19=high(2).
# npz encode order is 0=healthy, 1=COVID-19, 2=symptomatic -> explicit map, never assume.
RANK: dict[int, int] = {0: 0, 1: 2, 2: 1}
RANK_INV: dict[int, str] = {0: "low (healthy)", 1: "med (symptomatic)", 2: "high (COVID-19)"}
LABELS_INV = {0: "healthy", 1: "COVID-19", 2: "symptomatic"}
print("RANK map:", {LABELS_INV[k]: v for k, v in RANK.items()})

cv_pool: list[int] = [i for s in (train_subset, val_subset) for i in s.indices]

# recording index: path -> pool positions; recordings are single-label (npz blocks are per class)
rec_idx: dict[str, list[int]] = {}
for p in cv_pool:
    rec_idx.setdefault(dataset.paths[p], []).append(p)
rec_rank_of: dict[str, int] = {}
for path, idxs in rec_idx.items():
    labs = {int(dataset.labels[i]) for i in idxs}
    assert len(labs) == 1, f"recording {path} has mixed labels {labs} — grouping assumption broken"
    rec_rank_of[path] = RANK[next(iter(labs))]

cv_labels: list[int] = [rec_rank_of[dataset.paths[i]] for i in cv_pool]  # stratify on RANKS, not raw labels
FOLDS: list[tuple[list[int], list[int]]] = stratified_recording_folds(
    [dataset.paths[i] for i in cv_pool], cv_labels, n_splits=N_FOLDS, seed=SEED
)

# runnable check: leakage + stratification
_va_recs = [{dataset.paths[cv_pool[p]] for p in va_pos} for _, va_pos in FOLDS]
assert sorted(p for _, va in FOLDS for p in va) == list(range(len(cv_pool))), "folds don't partition the pool"
assert sum(len(rs) for rs in _va_recs) == len(rec_idx), "a recording landed in two val folds"
for fi, (tr_pos, va_pos) in enumerate(FOLDS):
    tr_recs = {dataset.paths[cv_pool[p]] for p in tr_pos}
    assert not (tr_recs & _va_recs[fi]), f"fold {fi}: a recording is in both train and val"
    assert {cv_labels[p] for p in va_pos} == {0, 1, 2}, f"fold {fi}: val fold lost a rank"
    counts = np.bincount([cv_labels[p] for p in va_pos], minlength=3)
    print(f"fold {fi}: val rank counts " + str({RANK_INV[k]: int(v) for k, v in enumerate(counts)}))
print(f"pool: {len(cv_pool)} segments / {len(rec_idx)} recordings; test holdout: {len(test_subset.indices)} segments")

healthy: 20092 (73.8%)
COVID-19: 1267 (4.7%)
symptomatic: 5852 (21.5%)
total: 27211
RANK map: {'healthy': 0, 'COVID-19': 2, 'symptomatic': 1}
fold 0: val rank counts {'low (healthy)': 3614, 'med (symptomatic)': 1055, 'high (COVID-19)': 230}
fold 1: val rank counts {'low (healthy)': 3614, 'med (symptomatic)': 1055, 'high (COVID-19)': 230}
fold 2: val rank counts {'low (healthy)': 3613, 'med (symptomatic)': 1055, 'high (COVID-19)': 231}
fold 3: val rank counts {'low (healthy)': 3614, 'med (symptomatic)': 1055, 'high (COVID-19)': 230}
fold 4: val rank counts {'low (healthy)': 3614, 'med (symptomatic)': 1054, 'high (COVID-19)': 230}
pool: 24494 segments / 10682 recordings; test holdout: 2717 segments


In [6]:
# ---- Recording-level rank counts -> inverse-frequency weights ----
# Weights are measured at the RECORDING level (effective sample size = recordings;
# sibling segments are correlated) and refit per CV fold inside train_ordinal.
def rank_counts_for(positions: Sequence[int]) -> list[int]:
    """Recording-level rank counts (one per recording touching `positions`)."""
    counts = [0, 0, 0]
    for path in {dataset.paths[i] for i in positions}:
        counts[rec_rank_of[path]] += 1
    return counts

print("full-pool recording counts:", rank_counts_for(cv_pool))
print("  -> cumulative pos weights:", [round(x, 4) for x in coral_pos_weights(rank_counts_for(cv_pool)).tolist()])

full-pool recording counts: [7960, 2193, 529]
  -> cumulative pos weights: [2.9243, 19.1928]


In [7]:
_model = CoralOrdinal(DefaultCNN(num_classes=1))
# combined pipeline: 64 mel + 40 MFCC = 104 freq bins, 3 channels (spec, mfcc, delta)
# CORAL head: backbone -> shared scalar score -> (B, 2) cumulative logits (P(rank>low), P(rank>med))
summary(_model, input_size=(128, 3, 104, 32))

Layer (type:depth-idx)                        Output Shape              Param #
CoralOrdinal                                  [128, 2]                  2
├─DefaultCNN: 1-1                             [128, 1]                  --
│    └─Sequential: 2-1                        [128, 256, 7, 2]          --
│    │    └─DepthwiseSeparableBlock: 3-1      [128, 32, 52, 16]         353
│    │    └─DepthwiseSeparableBlock: 3-2      [128, 64, 26, 8]          4,704
│    │    └─DepthwiseSeparableBlock: 3-3      [128, 128, 13, 4]         17,600
│    │    └─DepthwiseSeparableBlock: 3-4      [128, 256, 7, 2]          67,968
│    └─Sequential: 2-2                        [128, 1]                  --
│    │    └─AdaptiveAvgPool2d: 3-5            [128, 256, 1, 1]          --
│    │    └─Flatten: 3-6                      [128, 256]                --
│    │    └─Linear: 3-7                       [128, 1]                  257
Total params: 90,884
Trainable params: 90,884
Non-trainable params: 0
Total mult-ad

In [ ]:
# ---- Ordinal (CORAL) trainer: cumulative logits P(rank>low), P(rank>med) ----
def train_ordinal(
    name: str,
    model_name: str = "default",
    num_epochs: int = num_epochs,
    patience: int = 3,
    min_epoch: int = 15,
) -> tuple[dict, dict]:
    """CORAL ordinal training on the CV cell's rebound train_subset/val_subset.
    Single aug config (both). Crash-proof: _latest autosaved every epoch, _best on
    val-loss improvement, early stop at `min_epoch` after `patience` flat epochs
    (epoch-15 guardrail). OOF predictions are ALWAYS collected from the best
    checkpoint, even on a fully-resumed run. Returns (history, oof) where
    oof = {path, true_rank, p1, p2} per val segment (sigmoid cumulative probs)."""
    history: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "train_F1": [],
                                       "val_loss": [], "val_acc": [], "val_F1": [],
                                       "val_auc_r1": [], "val_auc_r2": []}

    # inverse-frequency weights at the RECORDING level (effective n = recordings)
    pos_w = coral_pos_weights(rank_counts_for(train_subset.indices)).to(device)

    # rarity-scaled augment probs from train segment counts (raw labels)
    seg_labels = torch.tensor([int(dataset.labels[i]) for i in train_subset.indices])
    counts3 = torch.bincount(seg_labels, minlength=3).float()
    aug_probs: dict[int, float] = aug_probs_from_counts({k: float(counts3[k]) for k in range(3)})
    print(f"[{name}] augment_probs: " + ", ".join(f"{k}:{v:.2f}" for k, v in sorted(aug_probs.items())))

    criterion: nn.Module = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    use_sep: bool = (model_name == "branched")
    base = _make_model(model_name, num_classes=1)
    model: nn.Module = torch.compile(CoralOrdinal(base)).to(device) if device.type == "cuda" \
        else CoralOrdinal(base).to(device)

    optimizer: optim.AdamW = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scaler: torch.amp.GradScaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
    scheduler: optim.lr_scheduler.CosineAnnealingLR = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )

    epochs_done, history, best_val_loss, plateau = _resume_with_best(
        name, model, optimizer, scheduler, scaler, num_epochs)
    # fresh-template backfill guard: new-key history (old checkpoints predate ordinal)
    for k in ("val_auc_r1", "val_auc_r2"):
        history.setdefault(k, []).extend([float("nan")] * (len(history["val_loss"]) - len(history.get(k, []))))

    # per-run spec-aug pipeline; waveform aug always on (single "both" config)
    mel_train = SpectMFCCPipeline(
        transforms=SpecAugmentations(augment_probs=aug_probs).to(device), combind=not use_sep,
    ).to(device)

    train_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        AugmentedSubset(dataset, train_subset.indices, WaveformAugmentor(16_000, augment_probs=aug_probs)),
        BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True,
    )
    val_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        Subset(dataset, val_subset.indices), BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True,
    )

    def forward_features(waveforms: torch.Tensor, raw: torch.Tensor, mel):
        """Combined pipelines return one (B,3,F,T) image; separated return (spec, mfcc)."""
        feats = mel(waveforms, raw)
        if use_sep:
            spec, mfcc = feats
            if spec.ndim == 3:
                spec, mfcc = spec.unsqueeze(0), mfcc.unsqueeze(0)
            return spec, mfcc
        if feats.ndim == 3:
            feats = feats.unsqueeze(0)
        return feats

    def _safe_auc(y_true, y_score) -> float:
        try:
            return float(roc_auc_score(y_true, y_score))
        except ValueError:
            return float("nan")  # degenerate val fold (single class) — not a crash

    for epoch in range(epochs_done, num_epochs):
        model.train()
        running_loss = 0.0
        train_preds: list[int] = []
        train_ranks: list[int] = []

        for waveforms, labels in tqdm(train_loader, desc=f"[{name}] train {epoch+1}/{num_epochs}", leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            raw = labels.long().to(device, non_blocking=True)
            rank = torch.tensor([RANK[int(l)] for l in raw.tolist()], device=device)

            optimizer.zero_grad(set_to_none=True)
            # features in fp32 OUTSIDE autocast: TimeStretch/complex-spectrogram NaN under fp16
            feats = forward_features(waveforms, raw, mel_train)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(*feats) if use_sep else model(feats)  # (B, 2) cumulative logits
                loss = criterion(outputs, coral_targets(rank, device=device).to(device))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            train_preds.extend(ordinal_predict(outputs.float()).cpu().numpy())
            train_ranks.extend(rank.cpu().numpy())

        scheduler.step()
        train_acc = 100.0 * float(np.mean(np.asarray(train_ranks) == np.asarray(train_preds)))
        train_f1 = f1_score(train_ranks, train_preds, average="macro", zero_division=0)

        model.eval()
        val_loss = 0.0
        all_preds: list[int] = []
        all_ranks: list[int] = []
        all_p1: list[float] = []
        all_p2: list[float] = []

        with torch.inference_mode():
            for waveforms, labels in tqdm(val_loader, desc=f"[{name}] val {epoch+1}/{num_epochs}", leave=False):
                waveforms = waveforms.to(device, non_blocking=True)
                raw = labels.long().to(device, non_blocking=True)
                rank = torch.tensor([RANK[int(l)] for l in raw.tolist()], device=device)
                feats = forward_features(waveforms, raw, mel_eval_sep if use_sep else mel_eval)
                with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    outputs = model(*feats) if use_sep else model(feats)
                    loss = criterion(outputs, coral_targets(rank, device=device).to(device))
                probs = torch.sigmoid(outputs.float())
                all_preds.extend(ordinal_predict(outputs.float()).cpu().numpy())
                all_ranks.extend(rank.cpu().numpy())
                all_p1.extend(probs[:, 0].cpu().numpy())
                all_p2.extend(probs[:, 1].cpu().numpy())
                val_loss += loss.item()

        tr = np.asarray(all_ranks)
        val_macro_f1 = f1_score(tr, all_preds, average="macro", zero_division=0)
        val_auc_r1 = _safe_auc((tr > 0).astype(int), np.asarray(all_p1))
        val_auc_r2 = _safe_auc((tr > 1).astype(int), np.asarray(all_p2))
        train_epoch_loss = running_loss / len(train_loader)
        val_epoch_loss = val_loss / len(val_loader)
        val_accuracy = 100.0 * float(np.mean(tr == np.asarray(all_preds)))

        history["train_loss"].append(train_epoch_loss)
        history["val_loss"].append(val_epoch_loss)
        history["val_F1"].append(val_macro_f1)
        history["val_acc"].append(val_accuracy)
        history["val_auc_r1"].append(val_auc_r1)
        history["val_auc_r2"].append(val_auc_r2)
        history["train_acc"].append(train_acc)
        history["train_F1"].append(train_f1)

        state = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
        }
        _atomic_torch_save(state, _ckpt_path(name))
        if val_epoch_loss < best_val_loss - 1e-6:
            best_val_loss = val_epoch_loss
            plateau = 0
            _atomic_torch_save(state, _ckpt_best_path(name))
        else:
            plateau += 1

        print(f"[{name}] Epoch [{epoch+1}/{num_epochs}] -> "
              f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_acc:.2f}% | Train F1: {train_f1:.4f} | "
              f"Val Loss: {val_epoch_loss:.4f} | Val F1: {val_macro_f1:.4f} | "
              f"Val AUC r1: {val_auc_r1:.4f} | Val AUC r2: {val_auc_r2:.4f} | Val Acc: {val_accuracy:.2f}%")

        # epoch-15 guardrail: stop once past epoch 15 unless val loss is still improving
        if epoch >= min_epoch and plateau >= patience:
            print(f"=> [{name}] early stop @ epoch {epoch+1}: val_loss flat {patience} epochs (best {best_val_loss:.4f})")
            break

    trained_models_all[name] = model

    # ---- OOF collection: ALWAYS runs (fresh, resumed-mid, or fully-complete) ----
    best_path = _ckpt_best_path(name) if os.path.exists(_ckpt_best_path(name)) else _ckpt_path(name)
    best = torch.load(best_path, weights_only=True, map_location=device)
    model.load_state_dict(best["model_state_dict"])
    model.eval()
    oof_paths: list[str] = []
    oof_true: list[int] = []
    oof_p1: list[float] = []
    oof_p2: list[float] = []
    with torch.inference_mode():
        for waveforms, labels in tqdm(val_loader, desc=f"[{name}] oof", leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            raw = labels.long()
            feats = forward_features(waveforms, raw, mel_eval_sep if use_sep else mel_eval)
            outputs = model(*feats) if use_sep else model(feats)
            probs = torch.sigmoid(outputs.float())
            oof_true.extend(RANK[int(l)] for l in raw.tolist())
            oof_p1.extend(probs[:, 0].cpu().numpy())
            oof_p2.extend(probs[:, 1].cpu().numpy())
    # paths come from the base indices in fold order (loader is shuffle=False)
    oof_paths = [dataset.get_path(int(i)) for i in val_subset.indices]
    oof = {
        "path": np.asarray(oof_paths, dtype=object),
        "true_rank": np.asarray(oof_true, dtype=int),
        "p1": np.asarray(oof_p1, dtype=float),
        "p2": np.asarray(oof_p2, dtype=float),
    }
    print(f"=> [{name}] OOF collected: {len(oof_p1)} segments / {len(set(oof_paths))} recordings")
    return history, oof

In [9]:
# ==== Model factory setup + separate-stream pipelines for the branched model ====
from modules.nn import BranchedCoughCNN
from modules.nn.base_nn import ResNetModel

CV_MODELS: list[str] = ["default", "branched", "resnet"]  # each model gets its own 5-fold CV run

# separate spec/mfcc streams for the branched model (pipeline returns (spec, mfcc))
mel_train_plain_sep: SpectMFCCPipeline = SpectMFCCPipeline(combind=False).to(device)
mel_eval_sep: SpectMFCCPipeline = SpectMFCCPipeline(combind=False).to(device)
mel_eval_sep = torch.compile(mel_eval_sep) if device.type == "cuda" else mel_eval_sep


In [10]:
# ---- Helpers shared by train_ordinal: model factory + best-checkpoint resume ----
def _make_model(model_name: str, num_classes: int = 1) -> nn.Module:
    """Model factory by string. resnet = ResNet50 ImageNet weights, head swapped, full finetune."""
    if model_name == "default":
        return DefaultCNN(num_classes)
    if model_name == "branched":
        return BranchedCoughCNN(num_classes)
    if model_name == "resnet":
        return ResNetModel(num_classes, freeze_backbone=False)
    raise ValueError(f"unknown model_name {model_name!r} (default|branched|resnet)")

def _ckpt_best_path(name: str) -> str:
    return os.path.join(RUN_STATE_DIR, f"{name}_best.pth.tar")

trained_models_all: dict[str, nn.Module] = {}  # registry _resume_with_best / train_ordinal write into

def _resume_with_best(name: str, model: nn.Module, optimizer: optim.Optimizer,
                      scheduler: optim.lr_scheduler.LRScheduler,
                      scaler: torch.amp.GradScaler, num_epochs: int) -> tuple[int, dict, float, int]:
    """Load _latest checkpoint into model/opt/sched/scaler; derive best val_loss
    and trailing plateau length from history. Returns (epochs_done, history, best_vl, plateau)."""
    best_vl, plate = float("inf"), 0
    path = _ckpt_path(name)
    fresh: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "train_F1": [],
                                     "val_loss": [], "val_acc": [], "val_F1": [],
                                     "val_auc_r1": [], "val_auc_r2": []}
    if not os.path.exists(path):
        return 0, fresh, best_vl, plate
    ckpt = torch.load(path, weights_only=True, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    epochs_done: int = ckpt["epoch"] + 1  # 'epoch' is the 0-based index of the last COMPLETED epoch
    history: dict[str, list[float]] = ckpt["history"]
    for k in ("train_acc", "train_F1", "val_auc_r1", "val_auc_r2"):  # older checkpoints predate these keys
        history.setdefault(k, []).extend([float("nan")] * (len(history["val_loss"]) - len(history.get(k, []))))
    if history["val_loss"]:
        best_vl = min(history["val_loss"])
        for vl in reversed(history["val_loss"]):
            if vl <= best_vl + 1e-6:
                break
            plate += 1
    if epochs_done >= num_epochs:
        trained_models_all[name] = model
    print(f"=> [{name}] resumed at epoch {epochs_done + 1}/{num_epochs} (best val_loss {best_vl:.4f})" if epochs_done < num_epochs
          else f"=> [{name}] already complete ({num_epochs}/{num_epochs} epochs) - restored, no retrain")
    return epochs_done, history, best_vl, plate


In [ ]:
# ---- 5-fold CV: ordinal models, pooled OOF recording-level metrics ----
# Every pool recording is validated exactly once across the folds. Each fold's
# best-val-loss checkpoint feeds pooled out-of-fold segment probabilities, which
# are averaged per RECORDING before any metric is computed (segment-level
# numbers are never the headline). Logs append to logs_3class/ordinal_cv_experiment.txt.
import sys, time, atexit
from datetime import datetime
from zoneinfo import ZoneInfo

LOG_PATH_CV = r"logs_3class/ordinal_cv_experiment.txt"
_logf_cv = open(LOG_PATH_CV, "a", encoding="utf-8")
sys.stdout = _logf_cv
sys.stderr = _logf_cv
atexit.register(_logf_cv.close)

print(f"\n\n\n TIME OF RUN: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}")

HIST_COLS: list[str] = ["train_loss", "train_acc", "train_F1", "val_loss", "val_acc", "val_F1", "val_auc_r1", "val_auc_r2"]

oof_all: dict[str, dict[str, np.ndarray]] = {}  # model -> pooled per-segment OOF across folds
cv_rows: list[dict] = []
_orig = (train_subset, val_subset)
try:
    for mn in CV_MODELS:
        seg_acc: dict[str, list] = {"path": [], "true_rank": [], "p1": [], "p2": [], "fold": []}
        for fi, (tr_pos, va_pos) in enumerate(FOLDS):
            train_subset = Subset(dataset, [cv_pool[p] for p in tr_pos])
            val_subset = Subset(dataset, [cv_pool[p] for p in va_pos])
            key = f"cv__{mn}__f{fi}"
            print(f"\n=== fold {fi + 1}/{len(FOLDS)}  {mn} ===")
            t0 = time.time()
            hist, oof = train_ordinal(key, model_name=mn)
            best_ep = int(np.argmin(hist["val_loss"]))
            cv_rows.append({
                "model": mn, "fold": fi, "epochs": len(hist["val_loss"]),
                "val_loss": round(hist["val_loss"][best_ep], 4),
                "val_acc": round(hist["val_acc"][best_ep], 4),
                "val_F1": round(hist["val_F1"][best_ep], 4),
                "val_auc_r1": round(hist["val_auc_r1"][best_ep], 4),
                "val_auc_r2": round(hist["val_auc_r2"][best_ep], 4),
            })
            print(f"=> {key}: best val_loss {hist['val_loss'][best_ep]:.4f} @ epoch {best_ep + 1} ({time.time() - t0:.0f}s)")
            seg_acc["path"].extend(oof["path"])
            seg_acc["true_rank"].extend(oof["true_rank"])
            seg_acc["p1"].extend(oof["p1"])
            seg_acc["p2"].extend(oof["p2"])
            seg_acc["fold"].extend([fi] * len(oof["p1"]))
            pl.DataFrame({k: hist[k] for k in HIST_COLS}).write_csv(f"cv__{mn}__f{fi}_experiment_stats.csv")
        oof_all[mn] = {k: np.asarray(v) for k, v in seg_acc.items()}
finally:
    train_subset, val_subset = _orig

# persist OOF state so eval/figures survive a fresh kernel
def _save_oof_state(obj: dict) -> None:
    tmp = "run_state_3class/ordinal_oof_state.pt.tmp"
    torch.save(obj, tmp)
    os.replace(tmp, "run_state_3class/ordinal_oof_state.pt")

_save_oof_state({"oof_all": oof_all, "cv_rows": cv_rows})
print("=> OOF state saved to run_state_3class/ordinal_oof_state.pt")
pl.DataFrame(cv_rows).write_csv("cv_ordinal_folds.csv")
print("=> wrote cv_ordinal_folds.csv")


[cv__default__f0] train 1/60:   0%|          | 0/307 [00:48<?, ?it/s]

[cv__default__f0] val 1/60:   0%|          | 0/77 [00:37<?, ?it/s]

[cv__default__f0] train 2/60:   0%|          | 0/307 [00:00<?, ?it/s]

[cv__default__f0] val 2/60:   0%|          | 0/77 [00:00<?, ?it/s]

[cv__default__f0] train 3/60:   0%|          | 0/307 [00:00<?, ?it/s]

[cv__default__f0] val 3/60:   0%|          | 0/77 [00:00<?, ?it/s]

... (90 outputs left out)

[cv__default__f2] train 16/60:   0%|          | 0/307 [00:00<?, ?it/s]

[cv__default__f2] val 16/60:   0%|          | 0/77 [00:00<?, ?it/s]

[cv__default__f2] oof:   0%|          | 0/77 [00:00<?, ?it/s]

[cv__default__f3] train 1/60:   0%|          | 0/307 [00:44<?, ?it/s]

In [ ]:
# ---- Recording-level OOF eval per model (the headline numbers) ----
# fresh-kernel recovery: load OOF state persisted by the CV cell
if "oof_all" not in globals() or not oof_all:
    _st = torch.load("run_state_3class/ordinal_oof_state.pt", weights_only=False)
    oof_all, cv_rows = _st["oof_all"], _st["cv_rows"]
    print("=> OOF state restored from run_state_3class/ordinal_oof_state.pt")

def aggregate_recordings(oof: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    """Mean cumulative probs per recording; predicted rank via 0.5 crossings."""
    order: dict[str, list[int]] = {}
    for i, p in enumerate(oof["path"]):
        order.setdefault(p, []).append(i)
    rec_true: list[int] = []
    rec_p1: list[float] = []
    rec_p2: list[float] = []
    rec_paths: list[str] = []
    for path, idxs in order.items():
        ranks = {int(oof["true_rank"][i]) for i in idxs}
        assert len(ranks) == 1, f"recording {path} spans ranks {ranks}"
        rec_paths.append(path)
        rec_true.append(ranks.pop())
        rec_p1.append(float(np.mean(oof["p1"][idxs])))
        rec_p2.append(float(np.mean(oof["p2"][idxs])))
    rec_p1, rec_p2 = np.asarray(rec_p1), np.asarray(rec_p2)
    rec_pred = (rec_p1 > 0.5).astype(int) + (rec_p2 > 0.5).astype(int)
    return {"path": np.asarray(rec_paths, dtype=object), "true": np.asarray(rec_true, dtype=int),
            "pred": rec_pred, "p1": rec_p1, "p2": rec_p2}

EVAL_COLS: list[str] = ["model", "n_recordings", "macro_f1", "recall_low", "recall_med", "recall_high",
                        "auc_r1", "auc_r2", "macro_auc", "rank_mae"]
eval_rows: list[dict] = []
rec_level: dict[str, dict[str, np.ndarray]] = {}
for mn in CV_MODELS:
    rec = aggregate_recordings(oof_all[mn])
    rec_level[mn] = rec
    m = ordinal_oof_metrics(rec["true"], rec["pred"], rec["p1"], rec["p2"])
    eval_rows.append({"model": mn, "n_recordings": len(rec["true"]),
                      **{k: (round(v, 4) if v is not None else None) for k, v in m.items()}})
    cm = confusion_matrix(rec["true"], rec["pred"], labels=[0, 1, 2])
    print(f"\n=== OOF recording-level [{mn}] ({len(rec['true'])} recordings) ===")
    print(f"(rows=true ['low','med','high'], cols=pred)\n{cm}")
    print({k: m[k] for k in m})
pl.DataFrame(eval_rows).write_csv("ordinal_oof_eval.csv")
print("=> wrote ordinal_oof_eval.csv")


In [ ]:
# ---- Figures: per-fold histories, ordinal CMs, P(risk>med) distributions ----
os.makedirs("figures_3class", exist_ok=True)
RANK_NAMES: list[str] = ["low", "med", "high"]

for mn in CV_MODELS:
    # history: one val-loss line per fold
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plotted = False
    for fi in range(N_FOLDS):
        csv = f"cv__{mn}__f{fi}_experiment_stats.csv"
        if os.path.exists(csv):
            df = pl.read_csv(csv)
            ax.plot(df["val_loss"].to_list(), label=f"fold {fi}")
            plotted = True
    if plotted:
        ax.set_title(f"{mn}: val loss per CV fold"); ax.set_xlabel("epoch"); ax.legend()
        fig.tight_layout(); fig.savefig(f"figures_3class/ordinal_{mn}_history.png", dpi=200)
        plt.show(fig); plt.close(fig)

    # ordinal confusion matrix (recording level)
    rec = rec_level[mn]
    cm = confusion_matrix(rec["true"], rec["pred"], labels=[0, 1, 2])
    disp = ConfusionMatrixDisplay(cm, display_labels=RANK_NAMES)
    disp.plot(cmap="Blues")
    plt.title(f"{mn} - OOF recording-level confusion matrix")
    plt.savefig(f"figures_3class/ordinal_{mn}_cm.png", bbox_inches="tight", dpi=200)
    plt.show(); plt.close("all")

    # honest signal picture: per-recording P(rank>med) by true rank
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for r in range(3):
        ax.hist(rec["p2"][rec["true"] == r], bins=30, alpha=0.5, label=RANK_NAMES[r], range=(0, 1))
    ax.set_title(f"{mn}: per-recording P(rank>med) by true rank"); ax.set_xlabel("P(rank>med)"); ax.legend()
    fig.tight_layout(); fig.savefig(f"figures_3class/ordinal_{mn}_p2_hist.png", dpi=200)
    plt.show(fig); plt.close(fig)
print("figures written to figures_3class/")
